# Notebook 03: Raster Reclassification & Suitability Scoring

This notebook reclassifies the continuous and discrete spatial variables (NDVI, DEM, Slope, Distance to Water, and LULC) into integer suitability scores ranging from **1 (Least Suitable)** to **5 (Most Suitable)**.

## Objectives:
1. Implement continuous reclassification functions.
2. Implement discrete LULC category reclass mapping.
3. Export five reclassified GeoTIFF raster layers.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import rasterio

# Configure paths
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.append(str(PROJECT_ROOT))

from src.reclassification import reclassify_file

print('Environment ready!')

## 1. Define Reclassification Criteria & Run Scoring

We establish reclassification bins for continuous variables (NDVI, DEM, Slope, Water Distance) and a dictionary lookup mapping for LULC classes. We then save the reclassified rasters in `data/processed/Reclassified/`.

In [ ]:
clean_dir = PROJECT_ROOT / 'data' / 'processed' / 'Cleaned'
reclass_dir = PROJECT_ROOT / 'data' / 'processed' / 'Reclassified'
reclass_dir.mkdir(parents=True, exist_ok=True)

criteria = {
    'ndvi': {
        'bins': [-np.inf, 0.20, 0.35, 0.45, 0.55, np.inf],
        'scores': [1, 2, 3, 4, 5],
        'input': clean_dir / 'NDVI_Clean.tif',
        'output': reclass_dir / 'NDVI_Reclass.tif',
        'discrete': False
    },
    'slope': {
        'bins': [-np.inf, 5, 15, 25, 35, np.inf],
        'scores': [5, 4, 3, 2, 1],  # Steeper is less suitable
        'input': clean_dir / 'Slope_Clean.tif',
        'output': reclass_dir / 'Slope_Reclass.tif',
        'discrete': False
    },
    'dem': {
        'bins': [-np.inf, 300, 500, 700, 900, np.inf],
        'scores': [5, 4, 3, 2, 1],  # Higher is less suitable
        'input': clean_dir / 'DEM_Clean.tif',
        'output': reclass_dir / 'DEM_Reclass.tif',
        'discrete': False
    },
    'distance': {
        'bins': [-np.inf, 250, 500, 1000, 2000, np.inf],
        'scores': [5, 4, 3, 2, 1],  # Closer is more suitable
        'input': clean_dir / 'DistanceToWater_Clean.tif',
        'output': reclass_dir / 'Distance_Reclass.tif',
        'discrete': False
    },
    'lulc': {
        'mapping': {
            0: 3,  # Water
            1: 5,  # Trees
            2: 4,  # Grass
            3: 3,  # Flooded vegetation
            4: 2,  # Crops
            5: 4,  # Shrub
            6: 1,  # Built
            7: 1,  # Bare
            8: 1   # Snow/Ice
        },
        'input': clean_dir / 'DynamicWorld_Clean.tif',
        'output': reclass_dir / 'LULC_Reclass.tif',
        'discrete': True
    }
}

for name, c in criteria.items():
    if c['discrete']:
        reclassify_file(c['input'], c['output'], mapping=c['mapping'], is_discrete=True)
    else:
        reclassify_file(c['input'], c['output'], bins=c['bins'], scores=c['scores'], is_discrete=False)

print('All layers successfully reclassified!')

## 2. Visualize Reclassified Scores

Let's plot the resulting reclassified rasters to inspect the score distributions (1-5).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
layers = [
    ('NDVI Reclass', reclass_dir / 'NDVI_Reclass.tif'),
    ('DEM Reclass', reclass_dir / 'DEM_Reclass.tif'),
    ('Slope Reclass', reclass_dir / 'Slope_Reclass.tif'),
    ('Distance Reclass', reclass_dir / 'Distance_Reclass.tif'),
    ('LULC Reclass', reclass_dir / 'LULC_Reclass.tif')
]

for idx, (title, path) in enumerate(layers):
    ax = axes[idx // 3, idx % 3]
    with rasterio.open(path) as src:
        arr = src.read(1)
        im = ax.imshow(arr, cmap='RdYlGn', vmin=1, vmax=5)
        ax.set_title(title)
        ax.axis('off')
        
# Turn off the empty last subplot
axes[1, 2].axis('off')
fig.colorbar(im, ax=axes.ravel().tolist()[:5], label='Suitability Score (1-5)', shrink=0.8)
plt.show()